### Install necessary libraries ###

In [16]:
%pip install "datasets[audio]" librosa jiwer transformers torch torchaudio

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [17]:
import torch
import librosa
import jiwer
import transformers
from datasets import load_dataset
from transformers import pipeline

print("All imports successful!")
print("Python environment:", __import__("sys").executable)
print("CUDA available:", torch.cuda.is_available())

All imports successful!
Python environment: c:\Users\ashle\code-switching\.venv\Scripts\python.exe
CUDA available: False


In [18]:
from transformers import pipeline

device = 0 if torch.cuda.is_available() else -1

transcriber = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-base",
    device=device,
)

print("Whisper is ready!")

Loading weights: 100%|██████████| 245/245 [00:00<00:00, 548.03it/s]


Whisper is ready!


In [19]:
import sys
import torch
import importlib.metadata as metadata

print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("TorchCodec:", metadata.version("torchcodec"))
print("Datasets:", metadata.version("datasets"))

Python: c:\Users\ashle\code-switching\.venv\Scripts\python.exe
PyTorch: 2.15.0+cpu
TorchCodec: 0.17.0
Datasets: 5.1.0


In [20]:
from datasets import load_dataset

# dataset = load_dataset(
#     "PolyAI/minds14",
#     name="en-AU",
#     split="train"
# )

# Load a small subset of spanish-english code-switching dataset for testing

dataset = load_dataset(
    "ky552/cszs_es_en",
    split="train[:100]"
)



sample = dataset[0]
print(sample.keys())

dict_keys(['correct_audio', 'correct_transcription', 'correct_file', 'wrong_audio', 'wrong_transcription', 'wrong_file'])


In [21]:
audio = sample["correct_audio"]

result = transcriber({
    "raw": audio["array"],
    "sampling_rate": audio["sampling_rate"]
})

print("Reference:", sample["correct_transcription"])
print("Prediction:", result["text"])

Reference: Algunas armas use two tipos de munición.
Prediction:  Algunas armas y ustu tipos de munición.


In [22]:
print(dataset)
print(dataset.column_names)
print("Number of examples:", len(dataset))

Dataset({
    features: ['correct_audio', 'correct_transcription', 'correct_file', 'wrong_audio', 'wrong_transcription', 'wrong_file'],
    num_rows: 100
})
['correct_audio', 'correct_transcription', 'correct_file', 'wrong_audio', 'wrong_transcription', 'wrong_file']
Number of examples: 100


In [23]:
for i in range(5):
    example = dataset[i]
    print(f"Example {i}")
    print("Correct transcription:", example["correct_transcription"])
    print("Wrong transcription:", example["wrong_transcription"])
    print("-" * 50)

Example 0
Correct transcription: Algunas armas use two tipos de munición.
Wrong transcription: Algunas armas use tres tipos de munición.
--------------------------------------------------
Example 1
Correct transcription: Desde el principio, the proyecto estuvo plagued por la falta de experiencia en engineering.
Wrong transcription: Desde el principio, the proyecto estuvo plagued por la falta de experiencia en swimming.
--------------------------------------------------
Example 2
Correct transcription: El disco includes catorce tracks.
Wrong transcription: El includes catorce disco tracks.
--------------------------------------------------
Example 3
Correct transcription: Part of the directorio of the Sociedad Rural Argentina formó.
Wrong transcription: Part of the orange of the Sociedad Rural Argentina formó.
--------------------------------------------------
Example 4
Correct transcription: Situado en la provincia de Castellón, in the region of la Plana Alta.
Wrong transcription: Situ

In [24]:
import numpy as np

durations = []

for i in range(len(dataset)):
    audio = dataset[i]["correct_audio"]
    duration = len(audio["array"]) / audio["sampling_rate"]
    durations.append(duration)

print(f"Shortest clip: {min(durations):.2f} seconds")
print(f"Longest clip: {max(durations):.2f} seconds")
print(f"Average duration: {np.mean(durations):.2f} seconds")

Shortest clip: 1.72 seconds
Longest clip: 6.35 seconds
Average duration: 3.82 seconds


In [25]:
audio = dataset[0]["correct_audio"]

result = transcriber({
    "raw": audio["array"],
    "sampling_rate": audio["sampling_rate"]
})

print(result)

print("Actual transcription:")
print(dataset[0]["correct_transcription"])

print("\nModel prediction:")
print(result)

{'text': ' Algunas armas y ustu tipos de munición.'}
Actual transcription:
Algunas armas use two tipos de munición.

Model prediction:
{'text': ' Algunas armas y ustu tipos de munición.'}


In [26]:
from IPython.display import Audio, display

for i in range(5):
    sample = dataset[i]
    audio = sample["correct_audio"]

    print(f"Sample {i}")
    display(Audio(
        audio["array"],
        rate=audio["sampling_rate"]
    ))

    print("Actual:", sample["correct_transcription"])
    print()

Sample 0


Actual: Algunas armas use two tipos de munición.

Sample 1


Actual: Desde el principio, the proyecto estuvo plagued por la falta de experiencia en engineering.

Sample 2


Actual: El disco includes catorce tracks.

Sample 3


Actual: Part of the directorio of the Sociedad Rural Argentina formó.

Sample 4


Actual: Situado en la provincia de Castellón, in the region of la Plana Alta.



In [30]:
import pandas as pd

num_samples = min(50, len(dataset))

results = []

for i in range(num_samples):

    sample = dataset[i]
    audio = sample["correct_audio"]

    prediction = transcriber({
        "raw": audio["array"],
        "sampling_rate": audio["sampling_rate"]
    })

    results.append({
        "id": i,
        "actual": sample["correct_transcription"],
        "predicted": prediction["text"]
    })

df = pd.DataFrame(results)


In [31]:
# none of the predictions are exactly correct

df["exactly_correct"] = df["actual"] == df["predicted"]

In [33]:
from jiwer import wer

df["wer"] = [
    wer(actual, predicted)
    for actual, predicted in zip(
        df["actual"],
        df["predicted"]
    )
]

df.sort_values(by="wer", ascending=True, inplace=True)

display(df[["id", "actual", "predicted", "wer"]])

print("Average WER:", df["wer"].mean())


C:\Users\ashle\AppData\Local\Temp\ipykernel_21992\3768883996.py:11: Pandas4Warning: The inplace keyword in DataFrame.sort_values is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.sort_values(by="wer", ascending=True, inplace=True)


,id,actual,predicted,wer
31,31,La Empresa Nacional de Celulosa era its offici...,La Empresa Nacional de Celulosa era its offic...,0.000000
38,38,"A partir de ese moment, los sucesos discográfi...","A partir de ese moment, los sucesos discográf...",0.000000
32,32,Todos los sectores of the economy se desarroll...,Todos los sectores of de economis se desarrol...,0.142857
17,17,Se puede considerar que he's el precursor mito...,se puede considerar que gisa el precursor mit...,0.200000
0,0,Algunas armas use two tipos de munición.,Algunas armas y ustu tipos de munición.,0.285714
1,1,"Desde el principio, the proyecto estuvo plague...","Desde el principio, el proyecto estuvo plagad...",0.285714
33,33,Los episodios especiales deviate from the form...,los episodios especiales debate from de forma...,0.300000
28,28,His father was the doctor Luis de Aguilar and ...,His father was the doctor Luis de Aguilar Ang...,0.307692
34,34,"Al tener both parents Mexicanos, Gilbert posse...","Al tener both parents mexicanos, Gilbert Poce...",0.333333
49,49,"En la bomba equilibrada Vickers, the paletas f...","En la bomba equilibrada Vickers, de paletas f...",0.363636


Average WER: 0.5832721167721168


In [34]:
# Listen to the 5 best and 5 worst predictions

for i in [31, 38, 32, 17, 0, 6, 7, 12, 24, 19]:
    sample = dataset[i]
    audio = sample["correct_audio"]

    print(f"Sample {i}")
    display(Audio(
        audio["array"],
        rate=audio["sampling_rate"]
    ))

    print("Actual:", sample["correct_transcription"])
    print()

Sample 31


Actual: La Empresa Nacional de Celulosa era its official name.

Sample 38


Actual: A partir de ese moment, los sucesos discográficos no se stop.

Sample 32


Actual: Todos los sectores of the economy se desarrollaron ya que el frente civil prospered.

Sample 17


Actual: Se puede considerar que he's el precursor mitológico del robot.

Sample 0


Actual: Algunas armas use two tipos de munición.

Sample 6


Actual: Él is international with the soccer team of Sudáfrica.

Sample 7


Actual: The origen of the image es desconocido.

Sample 12


Actual: Este disco has songs like: Cielo azul, cielo nublado and Ungrateful love.

Sample 24


Actual: Durante su juventud she studied derecho en Berlín and Múnich.

Sample 19


Actual: Finally he finished his career with the Natal Sharks.

